## Segmentation

Input_csv: 400_reviews_cleaned.csv

api_key = "..."  # Get from https://console.mistral.ai

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from mistralai.client import Mistral
from tqdm import tqdm
from src.config import DATA_DIR, RESULT_DIR

load_dotenv()
# Install: pip install mistralai tqdm

client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

local_dir = RESULT_DIR / "implicite_segmentation"
local_dir.mkdir(parents=True, exist_ok=True)

def segment_with_api(review_text):
    """Use Mistral API for segmentation"""
    prompt = f"""Segment this text into sentences. Keep citations like "et al." intact. Output one sentence per line.

Text: {review_text}"""
    
    response = client.chat.complete(
        model="mistral-small-latest",  # or "mistral-large-latest"
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    sentences = response.choices[0].message.content.split('\n')
    return sentences


In [ ]:
import torch.utils.data
from hydra import initialize, compose
from omegaconf import DictConfig
import os

from sentence_transformers.util import similarity

from src.data.dataloader import data_loader
from src.data.dataset_aprdb import AprdbDataset

import src.config
from src.detector.cristal_anchor.fetch_academic_paper import extract_text_from_pdf

with initialize(version_base=None, config_path="../../conf"):
    cfg: DictConfig = compose(config_name="config",overrides=[
        "experiment/dataset/aprbd@experiment.dataset=aprdb_tt_test",
        "experiment.data_loader=1_4_true",
        "experiment.dataset.subset_size=10000",
        "experiment.dataset.with_pdf=False",
        "experiment.dataset.split='calibration'"])

print(cfg)
ds_test = AprdbDataset(config=cfg.experiment.dataset)

#print(cfg)

In [ ]:
import pandas as pd
rows = []
for i in range(len(ds_test)):
    item = ds_test[i]
    rows.append(item)
df = pd.DataFrame(rows)
df

In [ ]:

# Process CSV with progress bar

# Enable tqdm for pandas apply
tqdm.pandas(desc="Segmenting reviews")

df['sentences'] = df['content'].progress_apply(segment_with_api)

df.to_csv(RESULT_DIR / "implicite_segmentation" /"results.csv", index=False, encoding="utf-8")

print(type(df['sentences'].iloc[0]))
print("✅ Done")


In [ ]:
# Explode the list into rows
import pandas as pd
from src.config import RESULT_DIR
df = pd.read_csv(local_dir / "results.csv")
from ast import literal_eval
df['sentences'] = df['sentences'].apply(literal_eval)
print(type(df['sentences'].iloc[0]))


df_expanded = df.explode('sentences', ignore_index=True)

# The 'sentences' column now contains individual strings (not lists)

df_expanded['sentences'] = df_expanded['sentences'].str.strip()

# Drop any rows where sentence is empty
df_expanded = df_expanded[df_expanded['sentences'] != '']

# Save or inspect
df_expanded.to_csv(local_dir / "output.csv", index=False, encoding="utf-8")
df_expanded
print(type(df_expanded['sentences'].iloc[0]))
#df

In [ ]:
# RULE-BASED Speech Act classification.

import re

QUESTION_WORDS = r"^(why|what|how|when|where|who|which|is|are|do|does|did|can|could|would|should)\b"

def classify(sentence):
    s = str(sentence).strip()

    if not s:
        return "Neither"

    s_low = s.lower()

    # --- Neither: formatting / metadata / fragments ---
    if len(s.split()) <= 2:
        return "Neither"

    if re.match(r"^(figure|table|section|chapter|source|url|http|www)\b", s_low):
        return "Neither"

    if s.startswith("[") and s.endswith("]"):
        return "Neither"

    if s_low in {"yes", "no", "ok", "okay"}:
        return "Neither"

    # --- Question detection ---
    if s.endswith("?"):
        return "Question"

    # wh-questions
    if re.match(QUESTION_WORDS, s_low):
        return "Question"

    # auxiliary inversion (common in English questions)
    if re.match(r"^(is|are|do|does|did|can|could|would|should|will|won't|didn't)\b", s_low):
        return "Question"

    # --- Default ---
    return "Assertion"
df = pd.read_csv(local_dir / "output.csv")
df["rule_based_SA-label"] = df["sentences"].apply(classify)

classified_filepath = local_dir / "classified_sentences.csv"
df.to_csv(classified_filepath, encoding="utf-8")
df

## Scores de subjectivite avec Roberta_Model.

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

hf_token = os.getenv("HF_TOKEN")
model_path = "/home/cyaiche/Documents/models/final_model_definitivo"

tokenizer = AutoTokenizer.from_pretrained(model_path, token=hf_token, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, token=hf_token, local_files_only=True)

mapping = {
    1: "WORLD",
    2: "WORLD+VALUES",
    3: "VALUES",
    4: "TASTE+VALUES",
    5: "TASTE"
}

csv_file_path = classified_filepath
df = pd.read_csv(csv_file_path)

if 'sentences' not in df.columns:
    raise ValueError("CSV file must contain a 'sentence' column")

# --- NEW: ensure label column exists ---
if 'rule_based_SA-label' not in df.columns:
    raise ValueError("CSV file must contain a 'label' column")

# Add prediction columns
df['raw_prediction_1_5'] = None
df['rounded_prediction'] = None
df['predicted_label'] = None

# --- NEW: filter only Assertion rows ---
df_assertion = df[df['rule_based_SA-label'] == "Assertion"].copy()

for index, row in df_assertion.iterrows():
    text = row['sentences']

    inputs = tokenizer(text, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = outputs.logits.squeeze().item()

    # Convert back to 1–5 scale
    prediction_1_5 = prediction * 4 + 1

    rounded_prediction = round(prediction_1_5)
    rounded_prediction = max(1, min(5, rounded_prediction))

    mapped_label = mapping[rounded_prediction]

    # write back to original df using index
    df.at[index, 'raw_prediction_1_5'] = prediction_1_5
    df.at[index, 'rounded_prediction'] = rounded_prediction
    df.at[index, 'predicted_label'] = mapped_label

    print(f"Processed {index + 1}/{len(df_assertion)}: {text[:50]}...")

output_file_path = "mistral_sentences_degrees.csv"
df.to_csv(local_dir / output_file_path, index=False, encoding="utf-8-sig")

print(f"\n✅ Predictions saved to {output_file_path}")
print(f"📊 Processed {len(df_assertion)} Assertion sentences")
df

In [ ]:
specific_df = df[df["pair_id"] == "ICLR2017-413-bde8ed5cbb"]
specific_df